In [65]:
%load_ext autoreload
%autoreload 2

import logging
from pathlib import Path

import altair as alt
import polars as pl

from game.fleet_placement_methods import PLACEMENT_METHODS
from game.game_logger import GameLogger
from main import run_games_headless

GameLogger.setup(console_level=logging.WARNING)
alt.data_transformers.enable("vegafusion")

REGENERATE_CSV = False
IMG_DIR = Path("img")
OUTPUT_CSV = Path("pre_analysis.csv")
GAMES_PER_CONFIG = 1000
AGENT_TYPES = [
    "random",
    "hunt",
    "bayes",
    "q-agent",
]
PLACEMENT_ORDER = list(PLACEMENT_METHODS.keys())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [66]:
if REGENERATE_CSV:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    records = []
    configs = [(a, m) for a in AGENT_TYPES for m in PLACEMENT_METHODS]
    n = len(configs)
    for i, (agent_type, method) in enumerate(configs):
        print(f"[{i + 1}/{n}] {agent_type} / {method} {'...':20s}", end="\r")
        for game_id, game in enumerate(
            run_games_headless(agent_type, method, GAMES_PER_CONFIG)
        ):
            records.append(
                {
                    "agent_type": agent_type,
                    "placement_method": method,
                    "game_id": game_id,
                    **game,
                }
            )
    df = pl.DataFrame(records)
    df.write_csv(OUTPUT_CSV)
    print(f"\nSaved {len(df):,} rows to {OUTPUT_CSV}")
else:
    df = pl.read_csv(OUTPUT_CSV)
    print(f"Loaded {len(df):,} rows from {OUTPUT_CSV}")

df.show()

Loaded 36,000 rows from pre_analysis.csv


agent_type,placement_method,game_id,agent_won,turns,agent_hits,agent_sunk,player_hits,player_sunk
str,str,i64,bool,i64,i64,i64,i64,i64
"""random""","""random""",0,false,96,15,3,17,5
"""random""","""random""",1,false,94,16,4,17,5
"""random""","""random""",2,false,98,16,4,17,5
"""random""","""random""",3,false,94,15,3,17,5
"""random""","""random""",4,true,96,17,5,16,4


In [67]:
summary = (
    df.group_by(["agent_type", "placement_method"])
    .agg(
        pl.col("agent_won").mean().alias("win_rate"),
        pl.col("turns").mean().alias("avg_turns"),
        pl.col("agent_hits").mean().alias("avg_agent_hits"),
        pl.col("player_hits").mean().alias("avg_player_hits"),
        pl.col("agent_sunk").mean().alias("avg_agent_sunk"),
        pl.col("player_sunk").mean().alias("avg_player_sunk"),
        pl.col("game_id").count().alias("n_games"),
    )
    .with_columns(
        (pl.col("win_rate") * 100).round(1).alias("win_rate_pct"),
        pl.col("avg_turns").round(1),
    )
    .sort(["agent_type", "placement_method"])
)
summary.show()

agent_type,placement_method,win_rate,avg_turns,avg_agent_hits,avg_player_hits,avg_agent_sunk,avg_player_sunk,n_games,win_rate_pct
str,str,f64,f64,f64,f64,f64,f64,u32,f64
"""bayes""","""clustered""",1.0,42.7,17.0,7.143,5.0,0.407,1000,100.0
"""bayes""","""corners""",1.0,52.7,17.0,8.949,5.0,0.707,1000,100.0
"""bayes""","""dense_center""",1.0,34.8,17.0,6.016,5.0,0.287,1000,100.0
"""bayes""","""diagonal""",1.0,45.2,17.0,7.653,5.0,0.508,1000,100.0
"""bayes""","""edges""",1.0,51.6,17.0,8.79,5.0,0.692,1000,100.0


In [113]:
def chart_title(text: str) -> alt.TitleParams:
    return alt.TitleParams(text, fontSize=14, fontWeight="normal", anchor="middle")


# Heatmap: agent win rate by agent type x player placement method
heatmap = (
    alt.Chart(summary)
    .mark_rect()
    .encode(
        x=alt.X(
            "placement_method:N",
            title="Player Placement Method",
            axis=alt.Axis(labelAngle=-35),
        ),
        y=alt.Y("agent_type:N", title="Agent Type"),
        color=alt.Color(
            "avg_turns:Q",
            title="Average Turns",
            scale=alt.Scale(scheme="blues", domain=[100, 30]),
        ),
        tooltip=[
            alt.Tooltip("agent_type:N", title="Agent"),
            alt.Tooltip("placement_method:N", title="Placement"),
            alt.Tooltip("win_rate_pct:Q", title="Win Rate (%)", format=".1f"),
            alt.Tooltip("avg_turns:Q", title="Avg Turns", format=".1f"),
        ],
    )
    .properties(
        title=chart_title("Agent Performance by Type and Player Placement Method"),
        width=500,
        height=150,
    )
)

labels = (
    alt.Chart(summary)
    .mark_text(fontSize=11)
    .encode(
        x=alt.X("placement_method:N"),
        y=alt.Y("agent_type:N"),
        text=alt.Text("avg_turns:Q", format=".0f"),
        color=alt.condition(
            alt.datum.avg_turns < 60,
            alt.value("white"),
            alt.value("black"),
        ),
    )
)

c = heatmap + labels
c.save(IMG_DIR / "win_rate_heatmap.png")
c.show()

alt.LayerChart(...)

In [98]:
# Bar chart: avg turns by agent type (collapsed across placement methods)
turns_by_agent = df.group_by("agent_type").agg(
    pl.col("turns").mean().round(2).alias("avg_turns"),
    pl.col("turns").std().alias("std_turns"),
)

c = (
    alt.Chart(turns_by_agent)
    .mark_bar()
    .encode(
        x=alt.X("agent_type:N", title="Agent Type"),
        y=alt.Y("avg_turns:Q", title="Average Turns"),
        color=alt.Color("agent_type:N", legend=None),
        tooltip=[
            alt.Tooltip("agent_type:N", title="Agent"),
            alt.Tooltip("avg_turns:Q", title="Avg Turns", format=".2f"),
        ],
    )
    .properties(
        title=chart_title("Average Turns by Agent Type (all placements)"),
        width=300,
        height=250,
    )
)
c.save(IMG_DIR / "avg_turns_by_agent.png")
c.show()

alt.Chart(...)

In [120]:
# Grouped bars: avg agent hits vs player hits by agent type
hits_long = pl.concat(
    [
        df.select(
            [
                "agent_type",
                pl.col("agent_hits").alias("hits"),
                pl.lit("Agent").alias("side"),
            ]
        ),
        df.select(
            [
                "agent_type",
                pl.col("player_hits").alias("hits"),
                pl.lit("Player (random)").alias("side"),
            ]
        ),
    ]
)

hits_summary = hits_long.group_by(["agent_type", "side"]).agg(
    pl.col("hits").mean().alias("avg_hits")
)

c = (
    alt.Chart(hits_summary)
    .mark_bar()
    .encode(
        x=alt.X("agent_type:N", title="Agent Type"),
        y=alt.Y("avg_hits:Q", title="Average Hits per Game"),
        xOffset=alt.XOffset("side:N"),
        color=alt.Color("side:N", title="Side"),
        tooltip=[
            alt.Tooltip("agent_type:N", title="Agent"),
            alt.Tooltip("side:N", title="Side"),
            alt.Tooltip("avg_hits:Q", title="Avg Hits", format=".1f"),
        ],
    )
    .properties(
        title=chart_title("Average Hits per Game: Agent vs Random Player"),
        width=350,
        height=250,
    )
)
c.save(IMG_DIR / "avg_hits_by_agent.png")
c.show()

alt.Chart(...)